# Leukemia Detection - EfficientNet-B3 Transfer Learning

## Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

## Dataset Loading

In [ ]:
train_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/C-NMC_Leukemia/fold_0/fold_0'
val_dir   = '/kaggle/input/datasets/avk256/cnmc-leukemia/C-NMC_Leukemia/fold_1/fold_1'
test_dir  = '/kaggle/input/datasets/avk256/cnmc-leukemia/C-NMC_Leukemia/fold_2/fold_2'

## Data Preprocessing

In [ ]:
datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = datagen.flow_from_directory(
    train_dir, target_size=(224,224), batch_size=32,
    class_mode='binary', shuffle=True
)

val_gen = datagen.flow_from_directory(
    val_dir, target_size=(224,224), batch_size=32,
    class_mode='binary', shuffle=False
)

test_gen = datagen.flow_from_directory(
    test_dir, target_size=(224,224), batch_size=32,
    class_mode='binary', shuffle=False
)

## Model Building

In [ ]:
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.summary()

## Model Compilation

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Model Training

In [ ]:
history = model.fit(train_gen, epochs=10, validation_data=val_gen)

## Evaluation

In [ ]:
loss, acc = model.evaluate(test_gen)
print(f'Test Accuracy: {acc*100:.2f}%')
print(f'Test Loss: {loss:.4f}')

## Results Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.legend()

plt.show()

In [ ]:
# Confusion Matrix
test_gen.reset()
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['ALL','HEM'], yticklabels=['ALL','HEM'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_true, y_pred, target_names=['ALL','HEM']))